## Status (created 2026-07-30) -- READ BEFORE RUNNING

**Standalone ComBat baseline notebook.** Deliberately a SEPARATE notebook from
`batch_correct_then_cluster_baselines.ipynb` (not an added arm inside it) so the two can
be run in parallel Colab sessions without one's config/state stomping the other's --
same underlying helper module (`interpretable_ssl/evaluation/batch_correct_baselines.py`),
just a different `CORRECTION_METHODS` value (`['combat']` here vs. `['harmony']` there).

Results land in the exact same on-disk run folders/metrics.json format the other
notebook uses (`seacell_X_combat`, `leiden_X_combat_K{n}`), so the comparison tables
below can pull in Harmony/scPoli/scVI results too, if those have already been run --
no need to re-run anything from the other notebook to see them side by side here.

If you already have this notebook open in another Colab tab, close it (or File >
Revert) first -- Colab autosaving from a stale tab can silently overwrite this file.


# Rebuttal experiment: ComBat batch-correction-then-cluster baseline -- ComBat latent -> {SEACells, Leiden}

**Purpose.** Reviewer nG29 (Question 3): *"How would batch-correction preprocessing
affect the comparison to other baselines? It would be interesting to see how the
performance of scProto would compare to other baselines if batch effects were removed a
priori (for example with Harmony or ComBat)."* The Harmony half of this question is
already answered in `batch_correct_then_cluster_baselines.ipynb`. This notebook runs the
ComBat half: apply original ComBat (Johnson, Li & Rabinovic 2007 -- `scanpy.pp.combat`,
built into scanpy, no extra install) to each dataset's log-normalized expression, PCA the
corrected matrix, then run **both** SEACells and Leiden on the resulting embedding --
replacing the paper's original weak `scPoli + K-means`-only comparison with a proper
two-step baseline, exactly as requested.

**Why ComBat is a meaningfully different comparison than Harmony/scPoli/scVI:** those
three all correct via a learned or iteratively-clustered embedding. ComBat instead fits
one linear (shift, spread) correction per gene per batch directly on gene expression --
no embedding, no clustering step, no neural network. Two structural properties of that
correction are worth watching for in the results (see
`baseline-notes/combat-notes.md` for the full reasoning):

1. ComBat cannot distinguish a genuine batch artifact from a rare cell state that
   happens to be confined to one batch -- both look like "batch i has an unusual mean
   for this gene," and correcting one flattens the other too.
2. ComBat fits a SINGLE shift per gene per batch, applied uniformly to every cell in
   that batch, even though real batch effects are well documented to hit different cell
   types/states by different amounts.

Both point at the same place: the rare-cell coverage/homogeneity/F1 table below is the
sharpest test of whether either limitation actually shows up in these three datasets --
watch that table first, not just the headline modularity/purity numbers.


## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install -q scarches faiss-gpu-cu12 scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 216.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 189.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 215.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 221.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 134.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 711.2/711.2 kB 86.1 MB/s eta 0:00:00
   ━━

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 217.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.1
    Uninstalling numpy-2.5.1:
      Successfully uninstalled numpy-2.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
seacells 0.3.3 requires pyranges, which is not installed.
anndata 0.13.2 requires scipy!=1.17.0,>=1.14, but you have scipy 1.13.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.66.0 which is incompatible.
tsfresh 0.21.2 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy

In [ ]:
# IMPORTANT: restart the runtime after this cell before running the cells below --
# numpy/scipy/anndata are C-extension linked, an in-process upgrade alone won't
# reliably take effect on already-imported modules.
#
# Note: scarches/scvi-tools/harmonypy are installed above even though THIS notebook
# only runs ComBat (which is plain scanpy, sc.pp.combat, no extra package) --
# run_all_baselines_for_dataset always calls get_stage1_latent() first to build `ad`
# from the existing scPoli Stage-1 checkpoint, and that path needs scarches/scvi
# importable regardless of which correction method is requested afterward.


In [ ]:
# Ground truth for "did the install cell above actually work" -- pip's own log is
# noisy (resolver backtracking prints "Getting requirements to build wheel" errors
# for discarded candidate versions even on a fully successful install), so eyeballing
# it is unreliable. Actually importing every package we just installed is the real
# test.
_checks = {
    'numpy': 'numpy', 'scipy': 'scipy', 'anndata': 'anndata', 'scanpy': 'scanpy',
    'scarches': 'scarches', 'scvi-tools': 'scvi', 'seacells': 'SEACells',
    'palantir': 'palantir', 'scib-metrics': 'scib_metrics', 'leidenalg': 'leidenalg',
    'python-igraph': 'igraph', 'umap-learn': 'umap', 'harmonypy': 'harmonypy',
    'faiss-cpu': 'faiss',
}
_failed = []
for pkg_name, import_name in _checks.items():
    try:
        mod = __import__(import_name)
        ver = getattr(mod, '__version__', '?')
        print(f"  OK   {pkg_name:16s} (import {import_name}, version {ver})")
    except Exception as e:
        _failed.append(pkg_name)
        print(f"  FAIL {pkg_name:16s} (import {import_name}): {type(e).__name__}: {e}")

if _failed:
    print(f"\n{len(_failed)} package(s) failed to import: {_failed} -- re-run that "
          f"package's specific pip install line above and check its full error "
          f"output before proceeding.")
else:
    print(f"\nAll {len(_checks)} packages import cleanly -- safe to continue.")


  OK   numpy            (import numpy, version 2.2.6)
  OK   scipy            (import scipy, version 1.13.1)


/tmp/ipykernel_2525/3189155525.py:17: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  ver = getattr(mod, '__version__', '?')
/tmp/ipykernel_2525/3189155525.py:17: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  ver = getattr(mod, '__version__', '?')


  OK   anndata          (import anndata, version 0.13.2)
  OK   scanpy           (import scanpy, version 1.12.3)


  FAIL scarches         (import scarches): ImportError: cannot import name 'read' from 'anndata' (/usr/local/lib/python3.12/dist-packages/anndata/__init__.py)
  OK   scvi-tools       (import scvi, version 1.5.0.post1)
  OK   seacells         (import SEACells, version 0.3.3)
  OK   palantir         (import palantir, version 1.4.5)
  OK   scib-metrics     (import scib_metrics, version 0.6.0)
  OK   leidenalg        (import leidenalg, version 0.12.0)
  OK   python-igraph    (import igraph, version 1.0.0)
  OK   umap-learn       (import umap, version 0.5.12)
  OK   harmonypy        (import harmonypy, version 2.0.0)
  OK   faiss-cpu        (import faiss, version 1.14.1)

1 package(s) failed to import: ['scarches'] -- re-run that package's specific pip install line above and check its full error output before proceeding.


In [1]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py


nb_setup done. Available: get_trainer, run_mc_task, fig_*, LAMBDA_PROTO_UMAP, LAMBDA_PROTO_UMAP_PRECON, LAMBDA_PARAM_UMAP, LAMBDA_RECON_ONLY, train_sure, eval_sure_task1/2/3
Configs: {'LAMBDA_PROTO_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'proto'}, 'LAMBDA_PARAM_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'embedding'}, 'LAMBDA_RECON_ONLY': {'lambda_umap': 0, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 1, 'lambda_proto_recon': 0.0}}


In [2]:
# Extra imports not already covered by nb_setup.py (which already pulls in
# run_mc_task, eval_seacell_task1/2/3, load_task1_multi, show_table, clean_run_names,
# rare_celltype_purity_table, TASK1_METRICS, TASK2_METRICS, extract_model_key via
# `from interpretable_ssl.evaluation.paper_figures import *` and
# `from interpretable_ssl.evaluation.metric_helpers.result_tables import *`).
from interpretable_ssl.datasets.dataset_configs import DATASETS
from interpretable_ssl.configs.paths import get_dataset_model_dir

print("extra imports ready")


extra imports ready


## Config

Same hyperparameters `train_scproto.ipynb` used to produce the existing Stage-1
checkpoints for these three datasets (`cvae_epochs=50`, `batch_size=1024`) -- must
match, or `load_pretrain_checkpoint()` looks in the wrong folder. `K` (n_SEACells /
target Leiden cluster count) defaults to each dataset's `num_prototypes` from
`DATASETS`, matching `results.tex`: *"All baselines are configured to produce the same
number of metacells K as scProto."*


In [3]:
RNA_SEQ_DATASETS = ['pancreas', 'lung', 'pbmc-immune']

SKIP_IF_EXISTS = True  # skip a baseline entirely (embedding + SEACells + Leiden) if its
                        # metrics.json already exists on disk -- re-running the dataset
                        # cell after a crash/interrupt never redoes already-saved work.

CORRECTION_METHODS = ['combat']  # this notebook: ComBat only (see intro cell)
METHOD_DISPLAY_NAMES = {
    'combat': 'ComBat',
}

RUN_SEACELLS = True  # both SEACells AND Leiden on top of ComBat -- reviewer nG29's Q3
                      # asked for the comparison itself, not one clusterer in isolation.
RUN_LEIDEN = True


## Helper functions

Shared with the Harmony notebook -- lives in
`interpretable_ssl/evaluation/batch_correct_baselines.py` (imported below), not
redefined here. ComBat support (`get_combat_corrected_pca`, and the `'combat'` branch
in `run_correction_method`) was added to that same module, so no separate helper file
exists for this notebook -- fixing a bug in the shared pipeline benefits both notebooks
at once, avoiding the exact duplication risk the Harmony notebook's own intro cell
describes from when it used to be split in two.


In [4]:
from interpretable_ssl.evaluation.batch_correct_baselines import run_all_baselines_for_dataset

print("baseline helper functions ready (interpretable_ssl.evaluation.batch_correct_baselines)")


baseline helper functions ready (interpretable_ssl.evaluation.batch_correct_baselines)


## Run: Pancreas

In [5]:
pancreas_results = run_all_baselines_for_dataset(
    'pancreas', correction_methods=CORRECTION_METHODS, skip_if_exists=SKIP_IF_EXISTS,
    run_seacells=RUN_SEACELLS, run_leiden=RUN_LEIDEN,
)


loading pancreas data


 captum (see https://github.com/pytorch/captum).


✅ Already subsetted to HVGs (4000 genes).
dataset is None, loading pancreas
loading pancreas data
✅ Already subsetted to HVGs (4000 genes).
proto_umap_ds-panc_NP220_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [9]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

📊 Affinity: wdeg[min/mean/max]=5.446/23.916/92.201, effk_med=62.4, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/pancreas/pretrain/pretrain_ds-pancreas_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 'pancreas', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'tech'}
📊 EdgeDataset: 1155146 edges
   Weight range: [0.0320, 0.9688]
   umap_steps_per_epoch=

  0%|          | 0/16 [00:00<?, ?it/s]


=== [pancreas] batch-correction method: combat ===
[pancreas] combat_d8: cached embedding to /content/drive/MyDrive/models/pancreas/_cache_X_combat_d8_emb.npy
Computing kNN graph using scanpy NN ...
Computing radius for adaptive bandwidth kernel...


...

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


  0%|          | 0/16382 [00:00<?, ?it/s]

Building similarity LIL matrix...


  0%|          | 0/16382 [00:00<?, ?it/s]

Constructing CSR matrix...
[pancreas] scProto (d=8): 0.454 +/- 0.184 (n=8 batches)
[pancreas] affinity purity saved to /content/drive/MyDrive/models/pancreas/rare_affinity_purity_pancreas.json


## Run: Lung

In [6]:
lung_results = run_all_baselines_for_dataset(
    'lung', correction_methods=CORRECTION_METHODS, skip_if_exists=SKIP_IF_EXISTS,
    run_seacells=RUN_SEACELLS, run_leiden=RUN_LEIDEN,
)


loading lung data
✅ Already subsetted to HVGs (4000 genes).
dataset is None, loading lung
loading lung data
✅ Already subsetted to HVGs (4000 genes).
proto_umap_ds-lung_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [16]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

📊 Affinity: wdeg[min/mean/max]=5.255/24.467/119.300, effk_med=63.8, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/lung/pretrain/pretrain_ds-lung_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 'lung', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'batch'}
📊 EdgeDataset: 2447924 edges
   Weight range: [0.0137, 0.9491]
   umap_steps_per_epoch=500 →

  0%|          | 0/32 [00:00<?, ?it/s]


=== [lung] batch-correction method: combat ===
[lung] combat_d8: cached embedding to /content/drive/MyDrive/models/lung/_cache_X_combat_d8_emb.npy
Computing kNN graph using scanpy NN ...
Computing radius for adaptive bandwidth kernel...


  0%|          | 0/32472 [00:00<?, ?it/s]

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


...

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


  0%|          | 0/32472 [00:00<?, ?it/s]

Building similarity LIL matrix...


  0%|          | 0/32472 [00:00<?, ?it/s]

Constructing CSR matrix...
[lung] scProto (d=8): 0.586 +/- 0.208 (n=15 batches)
[lung] affinity purity saved to /content/drive/MyDrive/models/lung/rare_affinity_purity_lung.json


## Run: PBMC (Immune)

In [7]:
immune_results = run_all_baselines_for_dataset(
    'pbmc-immune', correction_methods=CORRECTION_METHODS, skip_if_exists=SKIP_IF_EXISTS,
    run_seacells=RUN_SEACELLS, run_leiden=RUN_LEIDEN,
)


loading pbmc-immune data
✅ Already subsetted to HVGs (4000 genes).
dataset is None, loading pbmc-immune
loading pbmc-immune data
✅ Already subsetted to HVGs (4000 genes).
proto_umap_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [5]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

📊 Affinity: wdeg[min/mean/max]=1.667/25.923/299.545, effk_med=62.9, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/pbmc-immune/pretrain/pretrain_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 'pbmc-immune', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'study'}
📊 EdgeDataset: 2590828 edges
   Weight range: [0.0050, 0.9224]
   umap_ste

  0%|          | 0/33 [00:00<?, ?it/s]


=== [pbmc-immune] batch-correction method: combat ===
[pbmc-immune] combat_d8: cached embedding to /content/drive/MyDrive/models/pbmc-immune/_cache_X_combat_d8_emb.npy
Computing kNN graph using scanpy NN ...
Computing radius for adaptive bandwidth kernel...


  0%|          | 0/33506 [00:00<?, ?it/s]

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


...

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


  0%|          | 0/33506 [00:00<?, ?it/s]

Building similarity LIL matrix...


  0%|          | 0/33506 [00:00<?, ?it/s]

Constructing CSR matrix...
[pbmc-immune] scProto (d=8): 0.849 +/- 0.085 (n=5 batches)
[pbmc-immune] affinity purity saved to /content/drive/MyDrive/models/pbmc-immune/rare_affinity_purity_pbmc-immune.json


## Results comparison

Pulls in **every** existing run under `MODEL_DIR/{ds}/*` (scProto, SEACells(PCA), plus
whatever's already been computed by the Harmony notebook, alongside the new ComBat
baselines here) -- `load_task1_multi` / `rare_celltype_purity_table` just scan folders +
`metrics.json` / `umap_cells.csv`, they don't need to know which notebook produced them.

Harmony is included in the comparison tables below (via `HARMONY_DIM`) purely as a
READ of whatever's already on disk from the other notebook -- it is never (re)computed
here. If the Harmony notebook hasn't been run yet, those rows are simply absent, not an
error.

**Reminder:** the modularity column is scored against the arbf-on-PCA graph that only
scProto is trained to match -- treat it as a plausibility check, not the primary
evidence. Purity, batch entropy, and especially the rare-cell coverage/homogeneity
table (further down) are the fair, post-hoc comparison -- see the intro cell's note on
ComBat's two structural limitations for what to look for there specifically.


In [8]:
dataset_display_names = {'pancreas': 'Pancreas', 'lung': 'Lung', 'pbmc-immune': 'Immune'}

# scProto runs verified against the paper's published numbers via generate_tables.ipynb
# (the actual notebook used to build the paper's tables) -- bit-identical to the
# published numbers on multiple metrics simultaneously per dataset. Same three
# canonical run names the Harmony notebook uses (this is the SAME scProto model,
# just compared against a different baseline here).
SCPROTO_CANONICAL_RUNS = {
    'proto_umap_ds-panc_NP220_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31',
    'proto_umap_ds-lung_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31',
    'proto_umap_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31',
}

SCPROTO_KEY = extract_model_key(next(iter(SCPROTO_CANONICAL_RUNS)))
assert all(extract_model_key(r) == SCPROTO_KEY for r in SCPROTO_CANONICAL_RUNS), (
    "SCPROTO_CANONICAL_RUNS entries no longer normalize to one shared key -- "
    "extract_model_key's stripping patterns changed; update the scProto matching logic."
)

MODEL_KEYWORDS = {SCPROTO_KEY: 'scProto'}
MODEL_KEYWORDS['seacell'] = 'SEACells (PCA)'

# MATCHED_DIM must match scProto's actual latent_dims (currently 8 for all three
# datasets, per the Stage-1 pretrain params) -- a literal string match, not computed
# dynamically. ComBat is now ALSO dimension-matched (PCA'd to this dim instead of the
# old default 50) for the same reason Harmony already was: comparing scProto's 8-dim
# latent against a 50-dim-PCA-based baseline isn't apples-to-apples -- see
# batch_correct_baselines.DIM_MATCHED_METHODS / run_correction_method's
# matched_n_comps docstring for the full rationale. Old plain 'seacell_X_combat' /
# 'leiden_X_combat_K*' folders (pre-dimension-matching) are now orphaned leftovers,
# not read by this cell anymore.
MATCHED_DIM = 8
for _method in CORRECTION_METHODS:
    _disp = METHOD_DISPLAY_NAMES[_method]
    MODEL_KEYWORDS[f'seacell_X_{_method}_d{MATCHED_DIM}'] = f'SEACells ({_disp})'
    MODEL_KEYWORDS[f'leiden_X_{_method}_d{MATCHED_DIM}']  = f'Leiden ({_disp})'

# Harmony -- READ ONLY, pulled in for side-by-side comparison if the other notebook has
# already produced it (see 'Results comparison' markdown above).
MODEL_KEYWORDS[f'seacell_X_harmony_d{MATCHED_DIM}'] = 'SEACells (Harmony)'
MODEL_KEYWORDS[f'leiden_X_harmony_d{MATCHED_DIM}']  = 'Leiden (Harmony)'

In [9]:


def _keep_and_rename_runs(df, model_keywords=MODEL_KEYWORDS):
    # Filter load_task1_multi's output down to just the runs in model_keywords, and
    # collapse each method's per-dataset '_K{n}' folder-name suffix into ONE shared
    # display row per method. Identical logic to the Harmony notebook's version of this
    # helper -- kept as a local copy rather than imported, since these notebooks are
    # meant to run independently in parallel sessions.
    runs = df.index.get_level_values('run')
    datasets = df.index.get_level_values('dataset')
    stripped = runs.str.replace(r'_K\d+$', '', regex=True)
    keep_mask = stripped.isin(model_keywords)

    kept_runs, kept_datasets = runs[keep_mask], datasets[keep_mask]
    kept_display = stripped[keep_mask].map(model_keywords)

    out = df[keep_mask].copy()
    out.index = pd.MultiIndex.from_arrays([kept_datasets, kept_display], names=['dataset', 'run'])

    dupe_mask = out.index.duplicated(keep='first')
    if dupe_mask.any():
        stale = list(zip(kept_datasets[dupe_mask], kept_runs[dupe_mask], kept_display[dupe_mask]))
        print(f"WARNING: dropped {dupe_mask.sum()} duplicate (dataset, method) row(s) -- "
              f"likely a stale run at an old K value still on disk. "
              f"(dataset, on-disk folder, display name): {stale}")
        out = out[~dupe_mask]
    return out

# --- Table 1 (community structure / batch integration): modularity, batch entropy, purity ---
df_task1 = load_task1_multi(RNA_SEQ_DATASETS, metrics=TASK1_METRICS)
df_task1 = _keep_and_rename_runs(df_task1)
show_table(df_task1, metrics=TASK1_METRICS, dataset_display_names=dataset_display_names)


In [10]:
# --- Table 2 (metacell representation quality): coverage, scGraph (DGE consistency
# turned off, see intro) ---
df_task2 = load_task1_multi(RNA_SEQ_DATASETS, metrics=TASK2_METRICS)
df_task2 = _keep_and_rename_runs(df_task2)
show_table(df_task2, metrics=TASK2_METRICS, dataset_display_names=dataset_display_names)


In [11]:
# --- Rare-cell-type table (the key hypothesis test for BOTH of ComBat's structural
# limitations discussed in the intro cell): coverage + homogeneity + F1, same
# per-batch-rare-cell definition results.tex uses for Table 2 (tab:rare_cells).
#
# batch_rare_f1_macro_mean -- per-rare-type F1 (precision = purity of that type's
# dedicated metacell(s), recall = fraction of that type's cells that landed in one),
# macro-averaged across rare types, then mean+-std across batches.
#
# batch_rare_cross_batch_homog_mean -- same formula/denominator as homogeneity, but the
# numerator only counts same-label metacell-mates from a DIFFERENT batch -- directly
# tests whether rare types are actually grouped WITH their cross-batch counterparts,
# not just given their own same-batch-only cluster.
df_rare = rare_celltype_purity_table(RNA_SEQ_DATASETS, model_keywords=MODEL_KEYWORDS, verbose=True)

_dupe_mask = df_rare.index.duplicated(keep='first')
if _dupe_mask.any():
    print(f"WARNING: dropped {_dupe_mask.sum()} duplicate row(s) from df_rare: "
          f"{df_rare.index[_dupe_mask].tolist()}")
    df_rare = df_rare[~_dupe_mask]

show_table(
    df_rare,
    metrics=[
        'batch_rare_coverage_mean', 'batch_rare_recall_macro_mean',
        'batch_rare_precision_macro_mean', 'batch_rare_homogeneity_mean',
        'batch_rare_cross_batch_homog_mean', 'batch_rare_f1_macro_mean',
    ],
    dataset_display_names=dataset_display_names,
)


  [scProto|pancreas] resolving run dir ...
  [SEACells (PCA)|pancreas] resolving run dir ...
  [SEACells (ComBat)|pancreas] resolving run dir ...
  [SEACells (PCA)|pancreas] run dir resolved (0.0s)
  [Leiden (ComBat)|pancreas] resolving run dir ...
  [SEACells (Harmony)|pancreas] resolving run dir ...
  [Leiden (Harmony)|pancreas] resolving run dir ...
  [scProto|lung] resolving run dir ...
  [SEACells (PCA)|pancreas] reading umap_cells.csv ...
  [SEACells (PCA)|lung] resolving run dir ...
  [SEACells (PCA)|lung] run dir resolved (0.0s)
  [SEACells (PCA)|lung] reading umap_cells.csv ...
  [SEACells (PCA)|pancreas] umap_cells.csv loaded (16382 rows, 0.0s)
  [SEACells (PCA)|pancreas] reading umap_protos.csv ...
  [SEACells (PCA)|pancreas] umap_protos.csv loaded (0.0s)
  [SEACells (PCA)|pancreas] batch='tech' | 9 unique values, e.g. ['celseq', 'celseq2', 'fluidigmc1', 'inDrop1', 'inDrop2']
  [SEACells (PCA)|pancreas] label='celltype', batch='tech' | computing metacell label fractions ...


In [12]:
# --- PAIRED significance test for the rare-cell table above (Wilcoxon signed-rank,
# scProto vs. each same-K baseline, Bonferroni-corrected per dataset) -- same
# convention used throughout the Harmony notebook and results.tex.
from interpretable_ssl.evaluation.paper_figures import rare_metric_significance_paired

df_sig_paired = rare_metric_significance_paired(
    df_rare,
    ref_name='scProto',
    metrics=(
        '_batch_rare_f1_macro_per_batch',
        '_batch_rare_homogeneity_per_batch',
        '_batch_rare_cross_batch_homog_per_batch',
    ),
    dataset_display_names=dataset_display_names,
)
df_sig_paired


,dataset,metric,method,k,n,median,mean,std,n_wins,p_vs_ref,p_adj,sig
0,Pancreas,batch_rare_f1_macro,scProto,219,8,0.437680,0.535403,0.207961,NaN,NaN,NaN,NaN
1,Pancreas,batch_rare_f1_macro,SEACells (PCA),220,8,0.372628,0.370589,0.250869,6.0,0.039062,0.195312,ns
2,Pancreas,batch_rare_f1_macro,SEACells (ComBat),220,8,0.320318,0.345186,0.208526,7.0,0.007812,0.039062,*
3,Pancreas,batch_rare_f1_macro,Leiden (ComBat),220,8,0.149533,0.145890,0.066579,8.0,0.003906,0.019531,*
4,Pancreas,batch_rare_f1_macro,SEACells (Harmony),220,8,0.295521,0.316501,0.159421,7.0,0.011719,0.058594,ns
5,Pancreas,batch_rare_f1_macro,Leiden (Harmony),220,8,0.133361,0.223580,0.171156,8.0,0.003906,0.019531,*
6,Pancreas,batch_rare_homogeneity,scProto,219,8,0.522129,0.564854,0.155263,NaN,NaN,NaN,NaN
7,Pancreas,batch_rare_homogeneity,SEACells (PCA),220,8,0.369265,0.420592,0.195138,8.0,0.003906,0.019531,*
8,Pancreas,batch_rare_homogeneity,SEACells (ComBat),220,8,0.272370,0.291237,0.083672,8.0,0.003906,0.019531,*
9,Pancreas,batch_rare_homogeneity,Leiden (ComBat),220,8,0.208180,0.231644,0.101183,8.0,0.003906,0.019531,*


In [13]:
# --- PAIRED significance test for Table 1's modularity (scProto vs. each same-K
# baseline). Only modularity_per_batch is pairable (batch is a shared,
# method-independent unit) -- purity_per_mc / batch_entropy_per_mc stay unpaired
# (no natural cross-method correspondence between metacells from different methods).
from interpretable_ssl.evaluation.paper_figures import graph_batch_significance_paired

sig_df_table1_paired = graph_batch_significance_paired(
    RNA_SEQ_DATASETS,
    MODEL_KEYWORDS,
    ref_name='scProto',
    dataset_display_names=dataset_display_names,
)

sub = sig_df_table1_paired.copy()
sub['cell'] = sub.apply(
    lambda r: f"{r['median']:.3f} (K={r['k']}, n={r['n']}) [ref]" if r['method'] == 'scProto'
    else f"{r['median']:.3f} (K={r['k']}, n={r['n']}, wins={r.get('n_wins', '?')}/{r['n']})  "
         f"{r.get('sig', '?')}  p_adj={r.get('p_adj', float('nan')):.3g}",
    axis=1,
)
display(sub.pivot(index='method', columns='dataset', values='cell'))

sig_df_table1_paired


dataset,Immune,Lung,Pancreas
method,,,
Leiden (ComBat),"0.555 (K=300, n=5, wins=4.0/5) ns p_adj=0.781","0.481 (K=300, n=16, wins=16.0/16) *** p_adj=...","0.312 (K=220, n=9, wins=9.0/9) ** p_adj=0.00977"
Leiden (Harmony),"0.250 (K=300, n=5, wins=5.0/5) ns p_adj=0.156","0.403 (K=300, n=16, wins=16.0/16) *** p_adj=...","0.614 (K=220, n=9, wins=6.0/9) ns p_adj=1"
SEACells (ComBat),"0.588 (K=300, n=5, wins=3.0/5) ns p_adj=0.781","0.615 (K=300, n=16, wins=16.0/16) *** p_adj=...","0.570 (K=220, n=9, wins=7.0/9) ns p_adj=1"
SEACells (Harmony),"0.551 (K=300, n=5, wins=5.0/5) ns p_adj=0.156","0.625 (K=300, n=16, wins=14.0/16) ** p_adj=0...","0.566 (K=220, n=9, wins=7.0/9) ns p_adj=0.625"
SEACells (PCA),"0.569 (K=300, n=5, wins=5.0/5) ns p_adj=0.156","0.671 (K=300, n=16, wins=9.0/16) ns p_adj=1","0.658 (K=220, n=9, wins=0.0/9) ns p_adj=1"
scProto,"0.620 (K=294, n=5) [ref]","0.669 (K=298, n=16) [ref]","0.621 (K=219, n=9) [ref]"


,dataset,metric,method,k,n,median,mean,std,n_wins,p_vs_ref,p_adj,sig
0,Pancreas,modularity_per_batch,scProto,219,9,0.621456,0.601234,0.083385,NaN,NaN,NaN,NaN
1,Pancreas,modularity_per_batch,SEACells (PCA),220,9,0.657892,0.673906,0.050910,0.0,1.000000,1.000000,ns
2,Pancreas,modularity_per_batch,SEACells (ComBat),220,9,0.569886,0.578996,0.027181,7.0,0.212891,1.000000,ns
3,Pancreas,modularity_per_batch,Leiden (ComBat),220,9,0.311551,0.293305,0.066998,9.0,0.001953,0.009766,**
4,Pancreas,modularity_per_batch,SEACells (Harmony),220,9,0.566367,0.567150,0.017709,7.0,0.125000,0.625000,ns
5,Pancreas,modularity_per_batch,Leiden (Harmony),220,9,0.614318,0.612469,0.021231,6.0,0.544922,1.000000,ns
6,Lung,modularity_per_batch,scProto,298,16,0.669390,0.664444,0.022871,NaN,NaN,NaN,NaN
7,Lung,modularity_per_batch,SEACells (PCA),300,16,0.671340,0.673784,0.037789,9.0,0.628220,1.000000,ns
8,Lung,modularity_per_batch,SEACells (ComBat),300,16,0.615128,0.615392,0.025199,16.0,0.000015,0.000076,***
9,Lung,modularity_per_batch,Leiden (ComBat),300,16,0.480650,0.453209,0.084122,16.0,0.000015,0.000076,***


### Realized cluster/metacell count vs. scProto's target K

Sanity check that ComBat's downstream SEACells/Leiden actually landed at (approximately)
scProto's own `num_prototypes` for each dataset, per the paper's own protocol ("All
baselines are configured to produce the same number of metacells K as scProto"). Reads
directly from each run's saved outputs -- no recompute.


In [14]:
target_k = {ds: DATASETS[ds]['num_prototypes'] for ds in RNA_SEQ_DATASETS}

# Leiden: reads n_clusters/resolution straight from metrics.json
df_combat_k = load_task1_multi(RNA_SEQ_DATASETS, metrics=['n_clusters', 'resolution'])
if df_combat_k.empty:
    print("No runs found yet under MODEL_DIR for RNA_SEQ_DATASETS -- run the 'Run: ...' "
          "cells above first.")
else:
    is_combat_leiden = df_combat_k.index.get_level_values('run').str.startswith('leiden_X_combat_d8')
    df_combat_k_leiden = df_combat_k[is_combat_leiden].copy()
    if df_combat_k_leiden.empty:
        print("No leiden_X_combat_d8 run found yet for any dataset in RNA_SEQ_DATASETS.")
    else:
        df_combat_k_leiden['target_k'] = [target_k[ds] for ds, _run in df_combat_k_leiden.index]
        df_combat_k_leiden['matches_target'] = df_combat_k_leiden['n_clusters'] == df_combat_k_leiden['target_k']
        display(df_combat_k_leiden)

# SEACells: no K in the folder name -- read realized count from cell_assignments.csv directly
from interpretable_ssl.evaluation.batch_correct_baselines import get_realized_seacell_count

seacell_k_rows = []
for ds_id in RNA_SEQ_DATASETS:
    n_actual = get_realized_seacell_count(ds_id, 'X_combat_d8')
    k = target_k[ds_id]
    seacell_k_rows.append({
        'dataset': ds_id, 'method': 'combat',
        'n_actual': n_actual, 'target_k': k,
        'matches_target': (n_actual is not None and abs(n_actual - k) <= 0.05 * k),
    })

df_seacell_k = pd.DataFrame(seacell_k_rows).set_index(['dataset', 'method'])
if df_seacell_k['n_actual'].isna().all():
    print("No seacell_X_combat_d8 runs found on disk yet for RNA_SEQ_DATASETS.")
else:
    display(df_seacell_k)


,,n_clusters,resolution,target_k,matches_target
dataset,run,,,,
pancreas,leiden_X_combat_d8_K220,220.0,32.0,220,True
lung,leiden_X_combat_d8_K300,300.0,64.0,300,True
pbmc-immune,leiden_X_combat_d8_K300,300.0,64.0,300,True


,,n_actual,target_k,matches_target
dataset,method,,,
pancreas,combat,220,220,True
lung,combat,300,300,True
pbmc-immune,combat,300,300,True


## Embedding-only rare-cell affinity purity (no downstream clustering)

Isolates the embedding from whichever clustering algorithm runs on top of it -- for each
method's raw embedding, builds one ARBF affinity graph directly on it, then for each
locally-rare-type cell scores what fraction of its total affinity MASS goes to
same-type cells. This is the most direct empirical check of ComBat's structural
limitation #2 from the intro cell (one shift per gene per batch, not per cell type): if
that blunt correction is distorting rare states, it should show up here even before any
clustering algorithm gets a chance to compensate.

**No separate step to run here** -- this runs automatically at the end of
`run_all_baselines_for_dataset` (via `compute_and_save_embedding_affinity_purity`), so it
already ran as part of the `Run: Pancreas` / `Run: Lung` / `Run: PBMC (Immune)` cells
above. The cell below just loads and displays those saved results.


In [15]:
from interpretable_ssl.evaluation.batch_correct_baselines import load_and_compare_affinity_purity

df_affinity_purity = load_and_compare_affinity_purity(
    RNA_SEQ_DATASETS, dataset_display_names=dataset_display_names,
)
df_affinity_purity


,dataset,method,dim,n,mean,std,n_wins,p_vs_ref,p_adj,sig
0,Pancreas,Raw PCA (uncorrected),50,8,0.385,0.164,7.0,0.0742,0.1484,ns
1,Pancreas,ComBat,8,8,0.344,0.155,6.0,0.0547,0.1094,ns
2,Pancreas,scProto,8,8,0.454,0.184,NaN,NaN,NaN,NaN
3,Lung,Raw PCA (uncorrected),50,15,0.559,0.194,11.0,0.0240,0.0479,*
4,Lung,ComBat,8,15,0.399,0.171,14.0,0.0034,0.0067,**
5,Lung,scProto,8,15,0.586,0.208,NaN,NaN,NaN,NaN
6,Immune,Raw PCA (uncorrected),50,5,0.827,0.050,4.0,0.3125,0.6250,ns
7,Immune,ComBat,8,5,0.806,0.105,3.0,0.1562,0.3125,ns
8,Immune,scProto,8,5,0.849,0.085,NaN,NaN,NaN,NaN


## Targeted per-cell-type breakdown (combat-notes hypotheses)

The tables above macro-average rare-cell metrics across every locally-rare type
within a batch, then across batches -- which can mask exactly the kind of
per-type unevenness the two `baseline-notes/combat-notes.md` hypotheses predict.
This section checks both directly, targeting specific real cell types identified
empirically from the raw h5ad files (not synthetic/designed cases):

**Hypothesis 1 (ComBat confounds batch artifact with real rare biology confined to
one batch):** sharpest where a rare type lives almost entirely in ONE batch --
**Lung's `Neutrophils_IL1R2`** (472 cells, 6/16 batches, 93% concentrated in a
single batch) is the strongest case across all three datasets; Pancreas has no
comparably sharp case (its rare types spread across 8-9/9 batches).

**Hypothesis 2 (ComBat's one-shift-per-gene-per-batch can't fit every cell type
evenly):** the signature isn't the mean across rare types -- it's the SPREAD
(a method scoring great on one type and terrible on another can still average out
to look identical to a uniformly-mediocre method). Reported per type below, plus
a spread (std/min/max) summary across each dataset's bottom-quartile-frequency
rare types.

Reads already-saved `umap_cells.csv` per run -- no recompute, no retraining.


In [16]:
!ls -la /content/drive/MyDrive/codes/interpretable-prototype/interpretable_ssl/evaluation/ | grep rebuttal

-rw------- 1 root root  20157 Jul 30 14:52 rebuttal_report.py


In [17]:
from interpretable_ssl.evaluation.rebuttal_report import per_celltype_breakdown
import os
# Bottom-quartile-global-frequency rare types per dataset, identified directly from
# the raw h5ad files (obs['celltype']/['cell_type']/['final_annotation'] vs.
# obs['tech']/['batch']) -- see chat history for the per-batch concentration check
# that surfaced Neutrophils_IL1R2 as the sharpest hypothesis-1 case.
RARE_TYPES_BY_DATASET = {
    'pancreas': ['mast', 'epsilon', 'schwann', 't_cell'],
    'lung': ['Neutrophils_IL1R2', 'Type 1', 'Lymphatic', 'Ionocytes'],
    'pbmc-immune': ['Monocyte progenitors', 'Megakaryocyte progenitors', 'CD10+ B cells', 'Plasma cells'],
}

df_celltype_breakdown = pd.concat([
    per_celltype_breakdown(ds_id, MODEL_KEYWORDS, cell_types=cts)
    for ds_id, cts in RARE_TYPES_BY_DATASET.items()
], ignore_index=True)

df_celltype_breakdown


,dataset,method,cell_type,n_cells,homogeneity,precision,recall,f1
0,pancreas,scProto,mast,42,0.695,0.798,0.833,0.815
1,pancreas,scProto,epsilon,32,0.281,0.938,0.281,0.433
2,pancreas,scProto,schwann,25,0.628,0.791,0.720,0.754
3,pancreas,scProto,t_cell,7,0.212,0.000,0.000,0.000
4,pancreas,SEACells (PCA),mast,42,0.660,0.790,0.881,0.833
...,...,...,...,...,...,...,...,...
67,pbmc-immune,SEACells (Harmony),Plasma cells,129,0.489,0.923,0.473,0.625
68,pbmc-immune,Leiden (Harmony),Monocyte progenitors,428,0.314,0.422,0.315,0.361
69,pbmc-immune,Leiden (Harmony),Megakaryocyte progenitors,270,0.305,0.596,0.367,0.454
70,pbmc-immune,Leiden (Harmony),CD10+ B cells,207,0.629,0.736,0.758,0.747


In [18]:
# --- Hypothesis 1, headline case: Lung's Neutrophils_IL1R2 (93% in one batch) ---
# Does ComBat preserve it as a distinct metacell as well as scProto does, or does
# batch-correcting toward the cross-batch typical mean blur this batch-confined
# rare type's identity more than scProto's within-batch-affinity approach does?
df_celltype_breakdown[
    (df_celltype_breakdown['dataset'] == 'lung')
    & (df_celltype_breakdown['cell_type'] == 'Neutrophils_IL1R2')
].sort_values('f1', ascending=False)


,dataset,method,cell_type,n_cells,homogeneity,precision,recall,f1
28,lung,SEACells (PCA),Neutrophils_IL1R2,472,0.736,0.800,0.852,0.825
36,lung,Leiden (ComBat),Neutrophils_IL1R2,472,0.378,0.603,0.449,0.515
32,lung,SEACells (ComBat),Neutrophils_IL1R2,472,0.290,0.490,0.436,0.462
24,lung,scProto,Neutrophils_IL1R2,472,0.168,0.000,0.000,0.000
40,lung,SEACells (Harmony),Neutrophils_IL1R2,472,0.140,0.000,0.000,0.000
44,lung,Leiden (Harmony),Neutrophils_IL1R2,472,0.143,0.000,0.000,0.000


In [19]:
# --- Hypothesis 2: spread (not mean) across each dataset's rare types, per method ---
# High spread (max-min, or std) = uneven across cell types (predicted for ComBat's
# single global per-gene-per-batch shift); low spread = more uniform (predicted
# for scProto, which has no single global correction step).
df_spread = (
    df_celltype_breakdown
    .groupby(['dataset', 'method'])[['homogeneity', 'f1']]
    .agg(['mean', 'std', 'min', 'max'])
)
df_spread['homogeneity_range'] = df_spread[('homogeneity', 'max')] - df_spread[('homogeneity', 'min')]
df_spread['f1_range'] = df_spread[('f1', 'max')] - df_spread[('f1', 'min')]
df_spread


homogeneity                               f1  \
                                      mean       std    min    max     mean   
dataset     method                                                            
lung        Leiden (ComBat)        0.49450  0.095647  0.378  0.609  0.61750   
            Leiden (Harmony)       0.16150  0.104825  0.032  0.285  0.18025   
            SEACells (ComBat)      0.43325  0.332943  0.041  0.787  0.51525   
            SEACells (Harmony)     0.33325  0.134043  0.140  0.450  0.40525   
            SEACells (PCA)         0.80700  0.080440  0.736  0.918  0.88125   
            scProto                0.48075  0.415953  0.093  0.944  0.45025   
pancreas    Leiden (ComBat)        0.12725  0.140215  0.046  0.337  0.13675   
            Leiden (Harmony)       0.11775  0.096834  0.027  0.253  0.11375   
            SEACells (ComBat)      0.19300  0.144971  0.043  0.343  0.23925   
            SEACells (Harmony)     0.16275  0.119970  0.046  0.318  0.22000   
            SEACells (PCA)         0.32150  0.268105  0.045  0.660  0.34950   
            scProto                0.45400  0.242796  0.212  0.695  0.50050   
pbmc-immune Leiden (ComBat)        0.50725  0.262924  0.246  0.743  0.57400   
            Leiden (Harmony)       0.37500  0.171528  0.252  0.629  0.48150   
            SEACells (ComBat)      0.66275  0.132681  0.519  0.780  0.76050   
            SEACells (Harmony)     0.48400  0.100336  0.384  0.620  0.63075   
            SEACells (PCA)         0.79450  0.113621  0.682  0.913  0.85900   
            scProto                0.64325  0.234574  0.350  0.842  0.76225   

                                                       homogeneity_range  \
                                     std    min    max                     
dataset     method                                                         
lung        Leiden (ComBat)     0.098199  0.515  0.718             0.231   
            Leiden (Harmony)    0.250855  0.000  0.532             0.253   
            SEACells (ComBat)   0.382666  0.000  0.868             0.746   
            SEACells (Harmony)  0.271393  0.000  0.576             0.310   
            SEACells (PCA)      0.055271  0.825  0.956             0.182   
            scProto             0.523081  0.000  0.971             0.851   
pancreas    Leiden (ComBat)     0.273500  0.000  0.547             0.291   
            Leiden (Harmony)    0.227500  0.000  0.455             0.226   
            SEACells (ComBat)   0.278045  0.000  0.517             0.300   
            SEACells (Harmony)  0.257832  0.000  0.494             0.272   
            SEACells (PCA)      0.418136  0.000  0.833             0.615   
            scProto             0.373376  0.000  0.815             0.483   
pbmc-immune Leiden (ComBat)     0.311898  0.284  0.859             0.497   
            Leiden (Harmony)    0.182184  0.361  0.747             0.377   
            SEACells (ComBat)   0.092107  0.679  0.849             0.261   
            SEACells (Harmony)  0.070929  0.567  0.731             0.236   
            SEACells (PCA)      0.089088  0.766  0.955             0.231   
            scProto             0.241559  0.400  0.890             0.492   

                               f1_range  
                                         
dataset     method                       
lung        Leiden (ComBat)       0.203  
            Leiden (Harmony)      0.532  
            SEACells (ComBat)     0.868  
            SEACells (Harmony)    0.576  
            SEACells (PCA)        0.131  
            scProto               0.971  
pancreas    Leiden (ComBat)       0.547  
            Leiden (Harmony)      0.455  
            SEACells (ComBat)     0.517  
            SEACells (Harmony)    0.494  
            SEACells (PCA)        0.833  
            scProto               0.815  
pbmc-immune Leiden (ComBat)       0.575  
            Leiden (Harmony)      0.386  
            SEACells (ComBat)     0.170  
            SEACel

In [20]:

RARE_TYPES_BY_DATASET = {
    'pancreas': ['mast', 'epsilon', 'schwann', 't_cell'],
    'lung': ['Neutrophils_IL1R2', 'Type 1', 'Lymphatic', 'Ionocytes'],
    'pbmc-immune': ['Monocyte progenitors', 'Megakaryocyte progenitors', 'CD10+ B cells', 'Plasma cells'],
}

df_celltype_breakdown = pd.concat([
    per_celltype_breakdown(ds_id, MODEL_KEYWORDS, cell_types=cts)
    for ds_id, cts in RARE_TYPES_BY_DATASET.items()
], ignore_index=True)

df_celltype_breakdown

,dataset,method,cell_type,n_cells,homogeneity,precision,recall,f1
0,pancreas,scProto,mast,42,0.695,0.798,0.833,0.815
1,pancreas,scProto,epsilon,32,0.281,0.938,0.281,0.433
2,pancreas,scProto,schwann,25,0.628,0.791,0.720,0.754
3,pancreas,scProto,t_cell,7,0.212,0.000,0.000,0.000
4,pancreas,SEACells (PCA),mast,42,0.660,0.790,0.881,0.833
...,...,...,...,...,...,...,...,...
67,pbmc-immune,SEACells (Harmony),Plasma cells,129,0.489,0.923,0.473,0.625
68,pbmc-immune,Leiden (Harmony),Monocyte progenitors,428,0.314,0.422,0.315,0.361
69,pbmc-immune,Leiden (Harmony),Megakaryocyte progenitors,270,0.305,0.596,0.367,0.454
70,pbmc-immune,Leiden (Harmony),CD10+ B cells,207,0.629,0.736,0.758,0.747


In [21]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
display(df_celltype_breakdown)

save_path = '/content/drive/MyDrive/models/rare_celltype_breakdown_combat.csv'
df_celltype_breakdown.to_csv(save_path, index=False)
print(f"saved to {save_path}")

,dataset,method,cell_type,n_cells,homogeneity,precision,recall,f1
0,pancreas,scProto,mast,42,0.695,0.798,0.833,0.815
1,pancreas,scProto,epsilon,32,0.281,0.938,0.281,0.433
2,pancreas,scProto,schwann,25,0.628,0.791,0.720,0.754
3,pancreas,scProto,t_cell,7,0.212,0.000,0.000,0.000
4,pancreas,SEACells (PCA),mast,42,0.660,0.790,0.881,0.833
5,pancreas,SEACells (PCA),epsilon,32,0.045,0.000,0.000,0.000
6,pancreas,SEACells (PCA),schwann,25,0.397,0.619,0.520,0.565
7,pancreas,SEACells (PCA),t_cell,7,0.184,0.000,0.000,0.000
8,pancreas,SEACells (ComBat),mast,42,0.343,0.385,0.786,0.517
9,pancreas,SEACells (ComBat),epsilon,32,0.043,0.000,0.000,0.000


saved to /content/drive/MyDrive/models/rare_celltype_breakdown_combat.csv
